# Atividade Ponderada: MLP do Zero em NumPy

Este notebook documenta a construcao de um Multi-Layer Perceptron para classificar digitos do MNIST sem usar frameworks de deep learning para a matematica da rede. O objetivo aqui nao e esconder a complexidade atras de uma API pronta: e abrir o mecanismo em partes pequenas, validar cada parte e so depois escalar para o MNIST.

Resultado final registrado em `results/summary.json`: **97.59% de acuracia no teste** com arquitetura `784 -> 128 -> 64 -> 10`.


## Fontes de estudo usadas

Usei duas referencias iniciais para organizar o raciocinio:

- [Teaching a Perceptron by Hand](https://thomascountz.com/2018/03/26/perceptrons-implementing-and-part-1): ajudou a partir do caso mais simples, onde pesos e vies fazem uma decisao binaria.
- [Parameter optimization in neural networks](https://www.deeplearning.ai/ai-notes/optimization/index.html): fundamentou a separacao entre modelo, loss/custo, gradiente, learning rate e batch size.

A implementacao abaixo fica fiel a esse caminho: primeiro a transformacao linear `z = XW + b`, depois nao-linearidade, depois loss, gradientes e atualizacao dos parametros.


## Processo de desenvolvimento

1. Ler o enunciado e transformar requisitos em estrutura de repositorio.
2. Implementar o forward pass para qualquer numero de camadas.
3. Implementar softmax + cross-entropy de forma estavel.
4. Implementar backpropagation manual.
5. Validar gradientes numericamente em uma rede pequena.
6. Treinar duas configuracoes no MNIST e comparar resultados.
7. Registrar decisoes, dificuldades, resultados e evidencias no README.


In [ ]:
from pathlib import Path
import csv
import json
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mlp import MLPClassifier
from mlp.data import load_mnist
from mlp.losses import softmax

RESULTS = ROOT / 'results'
print('Repo:', ROOT)
print('Resultados disponiveis:', sorted(path.name for path in RESULTS.glob('*')))


## 1. Dados: MNIST como matriz NumPy

O enunciado permite carregar o MNIST via Keras ou Torchvision. Neste projeto, `torchvision.datasets.MNIST` faz apenas o download e a leitura dos arquivos. A partir dai, as imagens viram arrays NumPy `float32` normalizados entre 0 e 1.

Cada imagem `28 x 28` e achatada para um vetor de `784` atributos. A rede final recebe entao uma matriz `X` com forma `(n_exemplos, 784)` e rotulos `y` com valores de 0 a 9.


In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = load_mnist(ROOT / 'data', train_limit=1000, test_limit=200)
print('Treino:', X_train.shape, y_train.shape)
print('Validacao:', X_val.shape, y_val.shape)
print('Teste:', X_test.shape, y_test.shape)
print('Min/max dos pixels normalizados:', X_train.min(), X_train.max())


## 2. Arquitetura escolhida

A arquitetura final foi:

```text
784 entradas -> 128 ReLU -> 64 ReLU -> 10 logits -> softmax
```

A escolha usa duas camadas ocultas, como exigido no enunciado. ReLU foi escolhida por ser simples, barata e reduzir o problema de saturacao que aparece com sigmoid/tanh em redes maiores. A inicializacao usa escala de He nas camadas ocultas para manter a magnitude das ativacoes em uma faixa razoavel no comeco do treino.


In [ ]:
model = MLPClassifier([784, 128, 64, 10], activation='relu', seed=42)
logits = model.forward(X_train[:8])
probs = softmax(logits)
print('Logits:', logits.shape)
print('Probabilidades:', probs.shape)
print('Soma por exemplo:', probs.sum(axis=1))


## 3. Forward pass

Cada camada escondida calcula:

```text
Z[l] = A[l-1] @ W[l] + b[l]
A[l] = ReLU(Z[l])
```

A ultima camada nao aplica ReLU. Ela produz logits, que depois passam por softmax. Essa separacao evita misturar a camada linear final com a interpretacao probabilistica.


In [ ]:
import inspect
from mlp.network import MLPClassifier

print(inspect.getsource(MLPClassifier.forward))


## 4. Backpropagation e gradient check

O ponto mais sensivel foi garantir que os gradientes tinham as dimensoes e sinais corretos. Para softmax com cross-entropy, o gradiente dos logits fica:

```text
dZ_final = (probabilidades - y_one_hot) / batch_size
```

Depois disso, cada camada volta calculando `dW`, `db` e o gradiente que precisa ser enviado para a camada anterior. Para ganhar confianca, comparei alguns gradientes analiticos com a aproximacao numerica `(loss(W + eps) - loss(W - eps)) / (2 * eps)`.


In [ ]:
import numpy as np

rng = np.random.default_rng(13)
X_small = rng.normal(size=(5, 4)).astype(np.float32)
y_small = np.array([0, 1, 2, 1, 0])
small_model = MLPClassifier([4, 6, 5, 3], activation='tanh', seed=3)
loss, grads = small_model.loss_and_gradients(X_small, y_small)

name, index = 'W2', (1, 3)
epsilon = 1e-3
original = small_model.params[name][index]
small_model.params[name][index] = original + epsilon
loss_plus = small_model.loss(X_small, y_small)
small_model.params[name][index] = original - epsilon
loss_minus = small_model.loss(X_small, y_small)
small_model.params[name][index] = original

numerical = (loss_plus - loss_minus) / (2 * epsilon)
analytical = grads[name][index]
print('Loss pequena:', loss)
print('Gradiente analitico:', analytical)
print('Gradiente numerico:', numerical)
print('Erro absoluto:', abs(analytical - numerical))


## 5. Treinamento por mini-batches com SGD

O treinamento usa mini-batches de 128 exemplos. A cada batch:

1. forward pass calcula logits;
2. softmax + cross-entropy calcula loss;
3. backprop calcula gradientes;
4. SGD atualiza `W` e `b` na direcao oposta ao gradiente.

Usei decaimento multiplicativo simples no learning rate, porque um passo inicial maior acelera o comeco e passos menores reduzem oscilacao perto do minimo.

Para reproduzir o treino completo, rode no terminal a partir da raiz do repo:

```powershell
python scripts/train.py
```

O notebook le os resultados ja versionados para evitar retreinar tudo a cada abertura.


In [ ]:
summary_path = RESULTS / 'summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
for row in summary:
    print(f"{row['name']}: test_acc={row['test_accuracy']:.4f}, test_loss={row['test_loss']:.4f}, layers={row['layer_sizes']}")

assert max(row['test_accuracy'] for row in summary) >= 0.92


## 6. Comparacao de configuracoes

| Configuracao | Arquitetura | Epocas | LR inicial | Decaimento | Acuracia teste |
| --- | --- | ---: | ---: | ---: | ---: |
| `compact_relu_64_32` | 784 -> 64 -> 32 -> 10 | 8 | 0.08 | 0.96 | 96.34% |
| `final_relu_128_64` | 784 -> 128 -> 64 -> 10 | 10 | 0.12 | 0.96 | 97.59% |

A configuracao final teve mais capacidade e convergiu melhor, mas a compacta tambem passou da meta. Isso e um bom sinal: a implementacao nao depende de um hiperparametro fragil para funcionar.


## 7. Curvas de loss/acuracia

![Curvas de treino e validacao](../results/loss_accuracy.png)


## 8. Matriz de confusao

![Matriz de confusao do modelo final](../results/confusion_matrix_final.png)

Os erros restantes aparecem em pares visualmente parecidos, como digitos com tracos curvos ou inclinados. Isso e esperado em um MLP simples, porque ele nao explora estrutura espacial como uma CNN exploraria.


## 9. Decisoes e dificuldades

A decisao tecnica mais importante foi nao ir direto para o MNIST. Primeiro validei forward, depois gradient check, depois um problema pequeno, e so entao rodei MNIST. Isso evitou confundir erro de gradiente com problema de hiperparametro.

O que tentei e precisei ajustar:

- A leitura inicial do notebook falhou por sintaxe de shell: usei heredoc de Bash em PowerShell.
- A impressao do enunciado falhou por encoding `cp1252` ao encontrar simbolos matematicos.
- `tensorflow/keras` nao estava instalado no ambiente, entao usei Torchvision apenas para carregar MNIST.
- O stdout do treino completo ficou bufferizado quando rodei em processo separado; por isso usei arquivos de historico e `summary.json` como evidencia principal.

Se eu refizesse do zero, criaria desde o primeiro commit uma funcao de gradient check reutilizavel no proprio pacote, nao apenas nos testes. Tambem separaria melhor uma camada de experimentos para facilitar grid search maior sem mexer no notebook.


## 10. Conclusao

A entrega cumpre os requisitos obrigatorios: MLP com duas camadas ocultas, ReLU configuravel, softmax + cross-entropy, backpropagation manual, mini-batch SGD, comparacao de duas configuracoes, curvas de treino e acuracia final maior que 92% no teste.
